In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import sys
sys.path.append("/Users/samuel/Código/TFG")

# Cycle segmentation

In [10]:
import src.preprocess
import src.cycleDetection
import src.io
import pandas as pd
import librosa as lib
import numpy as np
import matplotlib.pyplot as plt

## Envelopes
These are the first step towards obtaining annotations on the cycles of each audio file. In order to not have the files being stored in disk from preprocessing and read again and for the purposes of synchronization (as I may forget to re-preprocess them in case I need to change the way(s) to do it), the preprocessing will be done here again. 

In [5]:
# Reading the inventory
inv = pd.read_csv("../data/interim/inventory.csv")

In [ ]:
def plot_envelope(band, sr, env, t_env, title="", xlim=None,
                  twin=True, mirror=False, band_label="heart band"):
    t_band = np.arange(len(band)) / sr

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t_band, band, color="grey", lw=0.5, alpha=0.5, label=band_label)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude")

    # Shannon: own y-axis; Hilbert: same axis as the band
    ax_env = ax.twinx() if twin else ax
    ax_env.plot(t_env, env, color="tab:blue", lw=1.5, label="envelope")
    if mirror:
        ax_env.plot(t_env, -env, color="tab:blue", lw=1.5)
    if twin:
        ax_env.set_ylabel("Envelope", color="tab:blue")
        ax_env.set_ylim(bottom=0)

    # One legend for both axes
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax_env.get_legend_handles_labels() if twin else ([], [])
    ax.legend(h1 + h2, l1 + l2, loc="upper right")

    ax.set_title(title)
    if xlim:
        ax.set_xlim(xlim)
    plt.tight_layout()
    plt.show()
# Same sort of loop as used for preprocessing. Will be used for creating the envelopes and segmenting. Segments will then be stored so they
# do not depend anymore on this loop.
conf = src.io.open_config()
for n, i in inv.iterrows():
    path = f"{conf["raw-data"]}/{i.path}"
    heart, lung = src.preprocess.pipeline(path, config=conf)

    # Hilbert envelope method is applied to lung band
    lung_envelope = src.cycleDetection.hilbert(lung, conf["sample-rate"], conf["hilbert-cutoff"])
    heart_envelope = src.cycleDetection.shannon(heart, conf["sample-rate"], conf["window-size"])
    plot_envelope(heart, conf["sample-rate"], heart_envelope, np.arange(len(heart_envelope))/conf["sample-rate"], title=i.file)
    